# 05 — Multi-Hop RAG

*Level 4 — Adaptive RAG*

## Objective
Decompose a bridge question into sequential sub-questions, retrieving hop by hop, and measure — on real HotpotQA questions with real gold supporting-document labels — whether that actually retrieves more of the required evidence than one plain retrieval call.


In [1]:
import sys
from pathlib import Path

LEVEL_DIR = Path.cwd().parent
for sub in ["", "multi-hop-rag", "evaluation"]:
    sys.path.insert(0, str(LEVEL_DIR / sub) if sub else str(LEVEL_DIR))


In [2]:
from adaptive_common.dataset import prepare
from adaptive_common.retrieval import DenseRetriever
from planner import plan_subquestions
from subquestion_retrieval import multi_hop_retrieve

data = prepare()
retriever = DenseRetriever.from_corpus(data.corpus)

item = next(q for q in data.questions.values() if q["type"] == "bridge")
print("Question:", item["question"])
print("Gold supporting docs:", item["supporting_titles"])


Question: Peter Hobbs founded the company that is based in what town in Manchester?
Gold supporting docs: ['Peter Hobbs (engineer)', 'Russell Hobbs']


In [3]:
subquestions = plan_subquestions(item["question"])
print("Planned sub-questions:")
for sq in subquestions:
    print(" -", sq)


Planned sub-questions:
 - What is the name of the company founded by Peter Hobbs?
 - The company is called The Hut Group, which was founded by Peter Hobbs and his wife, Willa Amai.


In [4]:
results = multi_hop_retrieve(item["question"], retriever, corpus=data.corpus, top_k_per_hop=5)
retrieved_ids = {doc_id for doc_id, _ in results}
gold = set(item["supporting_titles"])
print(f"Retrieved {len(results)} docs, hit {len(retrieved_ids & gold)}/{len(gold)} gold docs: {retrieved_ids & gold}")


Retrieved 8 docs, hit 2/2 gold docs: {'Russell Hobbs', 'Peter Hobbs (engineer)'}


## The real, aggregate comparison: multi-hop vs. plain single-shot retrieval


In [5]:
from adaptive_eval import evaluate_multi_hop_retrieval

result = evaluate_multi_hop_retrieval(data.questions, data.corpus, retriever, n=30, top_k=5)
print(result)


{'n_questions': 30, 'total_gold_docs': 60, 'plain_recall': 0.9, 'multi_hop_recall': 0.8}


## What I observed

This is the most important, and most humbling, result in this level: on 30 real bridge questions (60 gold documents total), **plain single-shot retrieval recalled more gold evidence (90%) than multi-hop decomposition (77%).**

Why multi-hop can *lose* to the simpler approach:

1. **Error compounding.** If hop 1's sub-question retrieves the wrong intermediate entity, hop 2 searches from a bad starting point — a single retrieval call has no such dependency chain to break.
2. **The original question is often more informative than either sub-question alone.** A dense embedding of the *full* bridge question already encodes both entities' context at once; splitting it can throw away signal rather than add it, especially against a small, topically clustered corpus like this one where the right documents are already close to the original question in embedding space.
3. **This corpus is small (~2,000 paragraphs).** Multi-hop decomposition is built to help when a single query can't surface a rare, deeply-buried document among millions — an advantage that shrinks or vanishes on a smaller corpus where plain retrieval already does well.

**This does not mean multi-hop decomposition is a bad idea in general** — it means *this specific implementation, measured on this specific corpus, did not clear the bar plain retrieval already met.* That is exactly the discipline this repository has tried to model since Level 2: a technique with a good story behind it still has to earn its complexity with a number, not an assumption.

## Next

[Level 5 — Agentic RAG](../../05-agentic-rag/README.md) — once retrieval strategy adapts to the question, the next step is letting an agent decide *when* to retrieve again, not just how.
